# Modelo ML XGBOOST CLASSIFIER

In [94]:
import pandas as pd
import numpy as np
import joblib
import json
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

# ========================
# A. Cargar datos
# ========================
X_train = pd.read_csv("../models/x_train_sel.csv")
y_train = pd.read_excel("../data/processed/X&Ys/y_train.xlsx").squeeze()

# ========================
# B. Cargar mapping ciudad (sin modificarlo)
# ========================
with open("../data/processed/Json/ciudad_transformation_rules.json") as f:
    ciudad_mapping = json.load(f)

id_to_ciudad = {str(v): k for k, v in ciudad_mapping.items()}
y_train_nombres = y_train.astype(str).map(id_to_ciudad)

# ========================
# C. Codificar ciudad
# ========================
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train_nombres)

# ========================
# D. Entrenar modelo con columnas actuales
# ========================
model = XGBClassifier(eval_metric="mlogloss")
model.fit(X_train, y_train_encoded)

# ========================
# E. Evaluar
# ========================
y_pred = model.predict(X_train)
print("✔️ Accuracy:", accuracy_score(y_train_encoded, y_pred))

# ========================
# F. Guardar modelo, encoder, columnas
# ========================
joblib.dump(model, "../models/model_completo.pkl")
joblib.dump(le, "../models/label_encoder.pkl")
X_train.columns.to_frame().to_csv("../data/processed/x_train_columns.csv", index=False, header=False)

✔️ Accuracy: 1.0


In [37]:
import joblib

joblib.dump(model, "../models/model_balanceado.pkl")
joblib.dump(le, "../models/label_encoder.pkl")
X_train.columns.to_frame().to_csv("../data/processed/x_train_columns.csv", index=False, header=False)

In [44]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
import joblib
import json

# ========================
# A. Cargar datos
# ========================
X_train = pd.read_csv("../models/x_train_sel.csv")
y_train = pd.read_excel("../data/processed/X&Ys/y_train.xlsx").squeeze()

# ========================
# B. Cargar mapping sin modificarlo
# ========================
with open("../data/processed/Json/ciudad_transformation_rules.json") as f:
    ciudad_mapping = json.load(f)  # formato: {nombre: id}

# Invertir el mapping para usarlo
id_to_ciudad = {str(v): k for k, v in ciudad_mapping.items()}
y_train_nombres = y_train.astype(str).map(id_to_ciudad)

# ========================
# C. Codificar nombres
# ========================
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train_nombres)

# ========================
# D. Asegurarse de que las columnas numéricas estén incluidas
# ========================
columnas_extra = [
    "estimated_price_eur_x",
    "estimated_price_eur_y",
    "distance_to_city_center_km",
    "class_n"
]

columnas_finales = columnas_extra + [
    col for col in X_train.columns
    if col not in columnas_extra  # Evita duplicados
]

X_train_final = X_train[columnas_finales]

# ========================
# E. Entrenar modelo
# ========================
model = XGBClassifier(use_label_encoder=False, eval_metric="mlogloss")
model.fit(X_train_final, y_train_encoded)

# ========================
# F. Evaluar
# ========================
y_pred = model.predict(X_train_final)
print("✔️ Accuracy:", accuracy_score(y_train_encoded, y_pred))

# ========================
# G. Guardar modelo y encoder
# ========================
joblib.dump(model, "../models/model_completo.pkl")
joblib.dump(le, "../models/label_encoder.pkl")
pd.Series(X_train_final.columns).to_csv("../data/processed/x_train_columns.csv", index=False, header=False)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/xgboost/core.py:158: UserWarning: [23:55:29] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


✔️ Accuracy: 1.0


# 🧪 PREDICCIÓN PERSONALIZADA

In [118]:
import pandas as pd
import numpy as np
import joblib
import json

# ========================
# 1. Cargar modelo y utilidades
# ========================
model = joblib.load("../models/model_completo.pkl")
le = joblib.load("../models/label_encoder.pkl")
columnas_modelo = pd.read_csv("../data/processed/x_train_columns.csv", header=None).squeeze().tolist()

with open("../data/processed/Json/ciudad_transformation_rules.json") as f:
    ciudad_mapping = json.load(f)
id_to_ciudad = {str(v): k for k, v in ciudad_mapping.items()}

full_df = pd.read_csv("../data/processed/total_data_240k.csv")

# ========================
# 2. Funciones auxiliares
# ========================
def construir_input_usuario(valores_dict, columnas_modelo):
    df = pd.DataFrame(columns=columnas_modelo)
    df.loc[0] = 0
    for clave, valor in valores_dict.items():
        col = f"{clave}_{valor}"
        if col in df.columns:
            df.at[0, col] = 1
    for col in ["estimated_price_eur_x", "estimated_price_eur_y", "distance_to_city_center_km"]:
        if col in valores_dict:
            df.at[0, col] = valores_dict[col]
    return df

def get_clima_estimado(ciudad, temporada):
    clima = full_df[(full_df['ciudad'] == ciudad) & (full_df['temporada'].str.lower() == temporada.lower())]
    if clima.empty:
        return "Sin datos"
    return clima[['temp_max', 'temp_min', 'precipitacion']].mean().round(1).to_dict()

def get_eventos(ciudad, temporada):
    eventos = full_df[
        (full_df['ciudad'] == ciudad) &
        (full_df['temporada'].str.lower() == temporada.lower())
    ][['evento_nombre', 'evento_categoria', 'evento_desc', 'fecha']].dropna().drop_duplicates().head(3)
    if eventos.empty or all(eventos['evento_nombre'].str.contains("sin_evento", case=False)):
        return "Sin eventos"
    return eventos.to_dict(orient='records')

def get_precio_vuelo(origen, destino):
    vuelos = full_df[(full_df['origin_city'] == origen) & (full_df['ciudad'] == destino)]
    return round(vuelos['flight_price'].mean(), 2) if not vuelos.empty else "Sin datos"

def get_hotel(ciudad):
    hoteles = full_df[full_df['ciudad'] == ciudad][['hotel_name', 'estimated_price_eur_y', 'hotel_type', 'distance_to_city_center_km']]
    hotel = hoteles.dropna().sort_values(by='estimated_price_eur_y').head(1)
    return hotel.to_dict(orient='records')[0] if not hotel.empty else "Sin hoteles"

# ========================
# 3. Entrada del usuario
# ========================
input_dict = {
    "perfil_viajero": "Soltero",
    "entornos": "Montaña",
    "clasificacion_destino": "Naturaleza",
    "temporada": "Invierno",
    "origin_city": "Madrid",
    "class": "Económica",
    "estimated_price_eur_x": 180,
    "estimated_price_eur_y": 95,
    "distance_to_city_center_km": 4.0
}

X_user = construir_input_usuario(input_dict, columnas_modelo)

# ========================
# 4. Predicción
# ========================
probs = model.predict_proba(X_user)[0]
top_indices = np.argsort(probs)[::-1]
top_labels = model.classes_[top_indices]
top_ciudades = [id_to_ciudad[str(lbl)] for lbl in top_labels]

# ========================
# 5. Mostrar con filtro
# ========================

print("🏝️ Top 5 ciudades recomendadas:")

mostradas = 0
i = 0
while mostradas < 5 and i < len(top_ciudades):
    ciudad = top_ciudades[i]
    i += 1

    clima = get_clima_estimado(ciudad, input_dict["temporada"])
    eventos = get_eventos(ciudad, input_dict["temporada"])
    vuelo = get_precio_vuelo(input_dict["origin_city"], ciudad)
    hotel = get_hotel(ciudad)

    if clima == "Sin datos" or eventos == "Sin eventos" or vuelo == "Sin datos" or hotel == "Sin hoteles":
        continue

    print(f"\n🌍 Ciudad: {ciudad}")
    print("☁️ Clima estimado:", clima)
    print("🎫 Eventos:", eventos)
    print("✈️ Vuelo desde origen:", vuelo)
    print("🏨 Hotel recomendado:", hotel)
    mostradas += 1

if mostradas == 0:
    print("\n⚠️ No se encontraron ciudades con información completa para los criterios dados.")

🏝️ Top 5 ciudades recomendadas:

🌍 Ciudad: honolulu
☁️ Clima estimado: {'temp_max': 18.4, 'temp_min': 10.1, 'precipitacion': 0.0}
🎫 Eventos: [{'evento_nombre': 'HANDLING-SAT MAT', 'evento_categoria': 'Desconocido', 'evento_desc': 'Sin descripción', 'fecha': '2025-12-13'}, {'evento_nombre': 'HANDLING-SAT EVE', 'evento_categoria': 'Desconocido', 'evento_desc': 'Sin descripción', 'fecha': '2025-12-13'}]
✈️ Vuelo desde origen: 1087.78
🏨 Hotel recomendado: {'hotel_name': 'Sunset Honolulu Residence', 'estimated_price_eur_y': 107.7, 'hotel_type': 'Hostal o Albergue', 'distance_to_city_center_km': 0.11}

🌍 Ciudad: miami
☁️ Clima estimado: {'temp_max': 18.4, 'temp_min': 10.1, 'precipitacion': 0.0}
🎫 Eventos: [{'evento_nombre': 'Playboi Carti', 'evento_categoria': 'Music', 'evento_desc': 'Unfortunately, the Event Organizer has had to cancel your event. No action is required to obtain a refund. It will be processed to the original method of payment used at time of purchase, once funds are receive